# DataSprint 2.0 — FUSE-Safe Kaggle to Google Drive Data Transfer (`train_images` & `val_images` Only)

Use a **free CPU Colab runtime** (no GPU needed). Before running:
1. Open [datasprint-2-road-damage-detection](https://www.kaggle.com/competitions/datasprint-2-road-damage-detection) on Kaggle and click **Join Competition** / accept the Rules.
2. Add `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab **Secrets (🔑)** (and toggle Notebook access ON), **or** keep `kaggle.json` ready to upload in Cell 2.

> **Scope & Durability:** Transfers **ONLY unarchived `train_images` and `val_images`** into `/content/drive/MyDrive/DataSprint_2.0_Road_Damage/01_data/`. Automatically skips `val_images` if `>= 2,150` images already exist, skips already-uploaded `train_images`, uses FUSE-safe batches (`workers=2`, `500`/batch + `os.sync()`), and runs `drive.flush_and_unmount()` + fresh remount verification so all files persist on Google Drive.


## 1. Mount Google Drive & Check Existing Image Counts

In [1]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DATA = Path('/content/drive/MyDrive/DataSprint_2.0_Road_Damage/01_data')
if not DRIVE_DATA.is_dir():
    raise FileNotFoundError(f'Missing Drive folder: {DRIVE_DATA}. Check Drive account or shortcut.')

for split_name in ('train_images', 'val_images'):
    (DRIVE_DATA / split_name).mkdir(parents=True, exist_ok=True)

def count_jpgs(folder: Path) -> int:
    return sum(1 for p in folder.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg'})

INITIAL_TRAIN = count_jpgs(DRIVE_DATA / 'train_images')
INITIAL_VAL = count_jpgs(DRIVE_DATA / 'val_images')
print(f'Existing Drive counts -> train_images: {INITIAL_TRAIN} | val_images: {INITIAL_VAL}')


Mounted at /content/drive
Existing Drive counts -> train_images: 2271 | val_images: 2150


## 2. Authenticate Kaggle (Colab Secrets or `kaggle.json` Upload)

In [2]:
import json, os, subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)

def _read_secret(secret_name: str):
    try:
        from google.colab import userdata
        val = userdata.get(secret_name)
        return val.strip() if isinstance(val, str) and val.strip() else None
    except Exception:
        return None

u_val = _read_secret('KAGGLE_USERNAME')
p_val = _read_secret('KAGGLE_KEY')
tok_val = _read_secret('KAGGLE_API_TOKEN')

if u_val and p_val:
    creds_payload = {'username': u_val, 'key': p_val}
    os.environ['KAGGLE_USERNAME'] = u_val
    os.environ['KAGGLE_KEY'] = p_val
    print('Loaded Kaggle credentials from Colab Secrets.')
elif tok_val:
    os.environ['KAGGLE_API_TOKEN'] = tok_val
    creds_payload = None
    print('Loaded KAGGLE_API_TOKEN from Colab Secrets.')
else:
    print('Colab Secrets not set — please upload your downloaded kaggle.json file:')
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No kaggle.json uploaded.')
    raw_bytes = next(iter(uploaded.values()))
    creds_payload = json.loads(raw_bytes.decode('utf-8'))
    os.environ['KAGGLE_USERNAME'] = creds_payload['username']
    os.environ['KAGGLE_KEY'] = creds_payload['key']

if creds_payload is not None:
    for cfg_dir in (Path('/root/.kaggle'), Path('/root/.config/kaggle')):
        cfg_dir.mkdir(parents=True, exist_ok=True)
        cfg_file = cfg_dir / 'kaggle.json'
        cfg_file.write_text(json.dumps(creds_payload), encoding='utf-8')
        cfg_file.chmod(0o600)
print('Kaggle CLI ready.')


Colab Secrets not set — please upload your downloaded kaggle.json file:


Saving kaggle.json to kaggle.json
Kaggle CLI ready.


## 3. Download Competition Zip & Selectively Extract Needed Splits (`train_images` / `val_images`)

In [3]:
import shutil, zipfile

NEED_VAL = INITIAL_VAL < 2150
NEED_TRAIN = INITIAL_TRAIN < 14691
TARGET_SPLITS = set()
if NEED_TRAIN:
    TARGET_SPLITS.add('train_images')
if NEED_VAL:
    TARGET_SPLITS.add('val_images')
print('Splits needing transfer:', sorted(TARGET_SPLITS) if TARGET_SPLITS else 'NONE (already complete)')

COMP_SLUG = 'datasprint-2-road-damage-detection'
DL_DIR = Path('/content/kaggle_dl')
EXT_DIR = Path('/content/kaggle_extracted')
TRAIN_SRC, VAL_SRC = [], []

if TARGET_SPLITS:
    for d in (DL_DIR, EXT_DIR):
        if d.exists():
            shutil.rmtree(d)
        d.mkdir(parents=True, exist_ok=True)

    subprocess.run(['kaggle', 'competitions', 'download', '-c', COMP_SLUG, '-p', str(DL_DIR)], check=True)
    zips = sorted(DL_DIR.glob('*.zip'))
    if not zips:
        raise FileNotFoundError(f'No zip downloaded in {DL_DIR}. Confirm you joined the competition and accepted its rules.')

    nested_targets = {f'{s}.zip' for s in TARGET_SPLITS}
    for zp in zips:
        with zipfile.ZipFile(zp, 'r') as zf:
            for info in zf.infolist():
                if info.is_dir():
                    continue
                parts = [pt.lower() for pt in Path(info.filename).parts]
                if any(s in parts for s in TARGET_SPLITS) and info.filename.lower().endswith(('.jpg', '.jpeg')):
                    zf.extract(info, EXT_DIR)
                elif Path(info.filename).name.lower() in nested_targets:
                    nested_zip = Path(zf.extract(info, DL_DIR))
                    target_split = nested_zip.stem
                    with zipfile.ZipFile(nested_zip, 'r') as nzf:
                        for ninfo in nzf.infolist():
                            if not ninfo.is_dir() and ninfo.filename.lower().endswith(('.jpg', '.jpeg')):
                                nzf.extract(ninfo, EXT_DIR / target_split)

    def _collect_split(split_name: str):
        return sorted({
            p.name: p
            for p in EXT_DIR.rglob('*')
            if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg'} and split_name in [pt.lower() for pt in p.parts]
        }.values())

    if NEED_TRAIN:
        TRAIN_SRC = _collect_split('train_images')
    if NEED_VAL:
        VAL_SRC = _collect_split('val_images')
    print(f'Extracted locally -> train_images: {len(TRAIN_SRC)} | val_images: {len(VAL_SRC)}')


Splits needing transfer: ['train_images']
Extracted locally -> train_images: 14691 | val_images: 0


## 4. FUSE-Safe Batched Copy + Drive Flush & Remount Verification

In [4]:
import threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed

def transfer_split_fuse_safe(split_name: str, src_files: list[Path], workers: int = 2, batch_size: int = 500):
    if not src_files:
        print(f'[{split_name}] skipped (already complete in Drive).')
        return
    dst_dir = DRIVE_DATA / split_name
    dst_dir.mkdir(parents=True, exist_ok=True)
    existing_sizes = {p.name: p.stat().st_size for p in dst_dir.iterdir() if p.is_file()}
    pending = [p for p in src_files if existing_sizes.get(p.name) != p.stat().st_size]
    skipped = len(src_files) - len(pending)
    print(f'[{split_name}] total={len(src_files)}, skipped_existing={skipped}, copying={len(pending)} (workers={workers}, batch={batch_size})...')
    if not pending:
        return

    t0 = time.monotonic()
    done = 0
    failed = []
    lock = threading.Lock()

    def _copy_with_retry(src: Path):
        tgt = dst_dir / src.name
        last_err = None
        for attempt in range(4):
            try:
                shutil.copy2(src, tgt)
                return True, src.name, None
            except OSError as exc:
                last_err = exc
                time.sleep(1.5 * (2 ** attempt))
        return False, src.name, str(last_err)

    for start_idx in range(0, len(pending), batch_size):
        chunk = pending[start_idx : start_idx + batch_size]
        with ThreadPoolExecutor(max_workers=workers) as pool:
            futs = [pool.submit(_copy_with_retry, p) for p in chunk]
            for fut in as_completed(futs):
                ok, name, err = fut.result()
                with lock:
                    done += 1
                    if not ok:
                        failed.append((name, err))
        os.sync()
        dt = max(0.01, time.monotonic() - t0)
        print(f'[{split_name}] {done}/{len(pending)} copied ({done / dt:.1f} img/s, failed={len(failed)}) — synced batch')
        time.sleep(2.0)
    if failed:
        raise RuntimeError(f'[{split_name}] {len(failed)} file(s) failed FUSE copy, e.g. {failed[:3]}. Rerun this notebook to resume.')

transfer_split_fuse_safe('val_images', VAL_SRC, workers=2, batch_size=500)
transfer_split_fuse_safe('train_images', TRAIN_SRC, workers=2, batch_size=500)
print('Skipped test_images and all non-image files (train & val images only).')

print('Flushing all Drive FUSE write buffers to Google Drive servers (do NOT close tab until this completes)...')
drive.flush_and_unmount()
print('Drive flush & unmount complete. Remounting to verify persisted counts...')
drive.mount('/content/drive', force_remount=True)

final_train = count_jpgs(DRIVE_DATA / 'train_images')
final_val = count_jpgs(DRIVE_DATA / 'val_images')
print(f'VERIFIED persisted Drive counts -> train_images: {final_train} | val_images: {final_val}')

shutil.rmtree(DL_DIR, ignore_errors=True)
shutil.rmtree(EXT_DIR, ignore_errors=True)
if final_train < 14691 or final_val < 2150:
    raise RuntimeError(f'Incomplete Drive counts after remount (train={final_train}/14691, val={final_val}/2150). Rerun to resume remaining files.')
print('ALL train_images (14,691) and val_images (2,150) verified on Google Drive!')


[val_images] skipped (already complete in Drive).
[train_images] total=14691, skipped_existing=2271, copying=12420 (workers=2, batch=500)...
[train_images] 500/12420 copied (49.5 img/s, failed=0) — synced batch
[train_images] 1000/12420 copied (50.1 img/s, failed=0) — synced batch
[train_images] 1500/12420 copied (50.2 img/s, failed=0) — synced batch
[train_images] 2000/12420 copied (49.8 img/s, failed=0) — synced batch
[train_images] 2500/12420 copied (49.5 img/s, failed=0) — synced batch
[train_images] 3000/12420 copied (49.3 img/s, failed=0) — synced batch
[train_images] 3500/12420 copied (49.0 img/s, failed=0) — synced batch
[train_images] 4000/12420 copied (48.9 img/s, failed=0) — synced batch
[train_images] 4500/12420 copied (48.8 img/s, failed=0) — synced batch
[train_images] 5000/12420 copied (48.5 img/s, failed=0) — synced batch
[train_images] 5500/12420 copied (48.2 img/s, failed=0) — synced batch
[train_images] 6000/12420 copied (48.1 img/s, failed=0) — synced batch
[train_i